In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: GEO02 - Business Travel
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.abac_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. AMEX Travel Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
   5. CWT Travel Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses a FULL OUTER JOIN between the Master AU List and the 
      Mapping Bootstrap. Strictly filtered by 'jurisdiction' for Canada, Hong Kong, 
      and Barbados.
   2. HIGH RISK JURISDICTIONS: Queries the ABAC risk table for 'High' ratings.
   3. TRAVEL DATA UNION: Unions AMEX and CWT files.
   4. FILTERING: INNER JOINS Travel Data to the High Risk Countries list.
   5. MAPPING & AGGREGATING: Joins high-risk trips to the CC Mapping table, 
      then SUMS the total number of trips per AU_ID.
   6. FINAL OUTPUT: Strict 6-column structure, converting NULL sums to '0', 
      and flagging if the AU is in the Master List.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data and FILTER BY TARGET COUNTRIES using 'jurisdiction'
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(business_segment) AS Business_Segment 
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has cost centers mapped to it
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine filtered Master List and Mapping
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    -- 4. Grab high-risk jurisdictions
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    -- 5a. AMEX data
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    
    UNION ALL
    
    -- 5b. CWT data
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
),

High_Risk_Trips AS (
    -- 6. Keep only high-risk destinations
    SELECT 
        t.Cleaned_CC,
        t.Trip_Count
    FROM Combined_Travel_Data t
    INNER JOIN High_Risk_Countries h
        ON t.DestinationCountry = h.CountryName
    WHERE t.Cleaned_CC IS NOT NULL
),

Trips_By_AU AS (
    -- 7. Map trips to AU_ID and sum
    SELECT 
        m.AU_ID,
        SUM(h.Trip_Count) AS Total_Trips
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN High_Risk_Trips h
        ON CAST(m.Cost_Center_ID AS INT) = h.Cleaned_CC
    GROUP BY m.AU_ID
)

-- 8. Final Template
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'GEO02' AS QuestionID,               
    COALESCE(CAST(t.Total_Trips AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Trips_By_AU t 
    ON a.Base_AU_ID = t.AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: GEO02 - Business Travel Drill-Down
=================================================================================== */

WITH Master_AUs AS (
    SELECT TRIM(CAST(business_ID AS STRING)) AS BusinessID
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    UNION ALL
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    t.Source_System,
    t.DestinationCountry AS High_Risk_Destination,
    t.Trip_Count
    
FROM vw_cost_center_mapping_bootstrap m

INNER JOIN Combined_Travel_Data t
    ON CAST(m.Cost_Center_ID AS INT) = t.Cleaned_CC
    
INNER JOIN High_Risk_Countries h
    ON t.DestinationCountry = h.CountryName
    
LEFT JOIN Master_AUs mast 
    ON m.AU_ID = mast.BusinessID
    
ORDER BY BusinessID;